## Part 1 — Document Corpus Setup

In [119]:
corpus = [
    "Transformers use self-attention mechanisms to encode relationships between words in a sequence.",
    "Attention allows models to weigh the importance of different tokens dynamically.",
    "The Transformer architecture replaces recurrence with parallel attention computation.",

    "Gradient descent is used to optimize neural network parameters by minimizing loss functions.",
    "Adam optimizer combines momentum and adaptive learning rates for faster convergence.",
    "Backpropagation computes gradients using the chain rule to update model weights.",

    "BERT is a bidirectional Transformer model pretrained on large corpora using masked language modeling.",
    "GPT models are autoregressive and generate text one token at a time.",

    "BM25 is a sparse retrieval method based on term frequency and inverse document frequency.",
    "Sentence Transformers generate dense vector embeddings for semantic similarity tasks.",

    "ColBERT uses late interaction and MaxSim scoring for efficient dense retrieval.",
]

## Part 2 — Implement Hybrid Retrieval

In [120]:
pip install rank_bm25

In [121]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import numpy as np

class HybridRetriever:
    def __init__(self, corpus, k=60):
        self.corpus = corpus
        self.k = k

        # BM25 setup
        self.tokenized_corpus = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(self.tokenized_corpus)

        # SBERT setup
        self.model = SentenceTransformer("all-MiniLM-L6-v2")
        self.embeddings = self.model.encode(corpus, convert_to_tensor=True)

    def retrieve(self, query, top_k=5):
        # BM25 scores
        tokenized_query = query.lower().split()
        bm25_scores = self.bm25.get_scores(tokenized_query)
        bm25_ranking = np.argsort(bm25_scores)[::-1]

        # SBERT scores
        query_embedding = self.model.encode(query, convert_to_tensor=True)
        sbert_scores = (self.embeddings @ query_embedding).cpu().numpy()
        sbert_ranking = np.argsort(sbert_scores)[::-1]

        # Rank lookup
        bm25_rank_dict = {doc_id: rank for rank, doc_id in enumerate(bm25_ranking)}
        sbert_rank_dict = {doc_id: rank for rank, doc_id in enumerate(sbert_ranking)}

        results = []

        for doc_id in range(len(self.corpus)):
            r_bm25 = bm25_rank_dict[doc_id]
            r_sbert = sbert_rank_dict[doc_id]

            rrf_score = (1 / (self.k + r_bm25)) + (1 / (self.k + r_sbert))

            results.append({
                "doc_id": doc_id,
                "rrf_score": rrf_score,
                "bm25_rank": r_bm25,
                "sbert_rank": r_sbert,
                "text": self.corpus[doc_id]
            })

        results = sorted(results, key=lambda x: x["rrf_score"], reverse=True)
        return results[:top_k]

## Part 3 — Implement a Cross-Encoder Re-Ranker

In [122]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, candidates, top_k=3):
    pairs = [(query, doc["text"]) for doc in candidates]

    scores = cross_encoder.predict(pairs)

    for i, doc in enumerate(candidates):
        doc["cross_score"] = scores[i]

    reranked = sorted(candidates, key=lambda x: x["cross_score"], reverse=True)

    return reranked[:top_k]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Part 4 — Implement Query Expansion

In [123]:
import google.generativeai as genai
import getpass

# Ask user for API key securely
API_KEY = getpass.getpass("Enter your Gemini API Key: ")

# Configure Gemini properly
genai.configure(api_key=API_KEY)

print("Gemini setup complete.")

Enter your Gemini API Key: ··········
Gemini setup complete.


In [124]:
def expand_query(query):
    try:
        # Gemini call (this may fail)
        model = genai.GenerativeModel("models/gemini-pro")

        prompt = f"""
        Generate exactly 3 different paraphrases of the following query.
        Return each on a new line without numbering.

        Query: {query}
        """

        response = model.generate_content(prompt)

        if not response.text:
            raise ValueError("Empty response")

        queries = response.text.strip().split("\n")
        return [q.strip("-•123456789. ") for q in queries if q.strip()][:3]

    except Exception as e:
        print("Gemini failed, using fallback:", e)

        return [
            query,
            f"explain {query}",
            f"{query} in machine learning context"
        ]

## Part 5 — End-to-End Pipeline

In [128]:
retriever = HybridRetriever(corpus)

def advanced_rag(user_query):
    # Step 1: Query Expansion
    expanded_queries = expand_query(user_query)[:2]

    all_docs = {}

    # Step 2: Retrieve for each query
    for q in expanded_queries:
        results = retriever.retrieve(q, top_k=5)

        for doc in results:
            all_docs[doc["text"]] = doc

    candidates = list(all_docs.values())

    # Step 3: Re-rank
    reranked_docs = rerank(user_query, candidates, top_k=3)

    context = "\n".join([doc["text"] for doc in reranked_docs])

    final_answer = f"""
    Question: {user_query}

    Answer based on retrieved documents:
    {context}
    """

    return final_answer

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Part 6 — Comparison Experiment

In [129]:
def naive_rag(query):
    model = SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = model.encode(corpus, convert_to_tensor=True)
    query_embedding = model.encode(query, convert_to_tensor=True)

    scores = (embeddings @ query_embedding).cpu().numpy()
    top_doc_id = np.argmax(scores)

    return corpus[top_doc_id]

In [130]:
import time

for q in queries:
    print("\nQuery:", q)
    print("Naive:", naive_rag(q))
    print("Advanced:", advanced_rag(q))

    time.sleep(20)


Query: how do transformers encode meaning?


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Naive: Transformers use self-attention mechanisms to encode relationships between words in a sequence.


Gemini failed, using fallback: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-pro:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-pro is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.
Advanced: 
    Question: how do transformers encode meaning?

    Answer based on retrieved documents:
    Transformers use self-attention mechanisms to encode relationships between words in a sequence.
Sentence Transformers generate dense vector embeddings for semantic similarity tasks.
The Transformer architecture replaces recurrence with parallel attention computation.
    

Query: optimization techniques for training


...

Gemini failed, using fallback: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-pro:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-pro is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.
Advanced: 
    Question: optimization techniques for training

    Answer based on retrieved documents:
    Adam optimizer combines momentum and adaptive learning rates for faster convergence.
Gradient descent is used to optimize neural network parameters by minimizing loss functions.
Backpropagation computes gradients using the chain rule to update model weights.
    

Query: what is bm25 used for?


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Naive: BM25 is a sparse retrieval method based on term frequency and inverse document frequency.


Gemini failed, using fallback: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-pro:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-pro is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.
Advanced: 
    Question: what is bm25 used for?

    Answer based on retrieved documents:
    BM25 is a sparse retrieval method based on term frequency and inverse document frequency.
Gradient descent is used to optimize neural network parameters by minimizing loss functions.
ColBERT uses late interaction and MaxSim scoring for efficient dense retrieval.
    


| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Are they different? |
|------|------------------|----------------------|---------------------|
| how do transformers encode meaning? | Transformers use self-attention... | Attention allows models to weigh importance... | Yes |
| optimization techniques for training | Gradient descent is used... | Adam optimizer combines momentum... | Yes |
| what is bm25 used for? | Sentence Transformers generate embeddings... | BM25 is a sparse retrieval method... | Yes |